# Week 9 Cross-Domain Robustness

This notebook is a lightweight launcher for the Week 9 jobs.
Run from repo root so relative paths match.

In [ ]:
import os
import subprocess

ROOT = os.path.abspath('..')
print(ROOT)

In [ ]:
import sys as _sys, os as _os

subprocess.run([
    _sys.executable,
    os.path.join(ROOT, 'scripts', 'jobs', 'job3_week9_eval.py'),
    '--fineweb-records',  os.path.join(ROOT, 'outputs', 'fineweb_labeled.json'),
    '--marco-records',    os.path.join(ROOT, 'outputs', 'marco_labeled.json'),
    '--scifact-records',  os.path.join(ROOT, 'outputs', 'scifact_sweep_records.json'),
    '--fiqa-records',     os.path.join(ROOT, 'outputs', 'fiqa_sweep_records.json'),
    '--scifact-data-path', os.path.join(ROOT, 'outputs', 'beir_data', 'scifact'),
    '--fiqa-data-path',   os.path.join(ROOT, 'outputs', 'beir_data', 'fiqa'),
    '--output-dir',       os.path.join(ROOT, 'outputs', 'week9'),
], check=True, cwd=ROOT, env={**_os.environ, 'PYTHONPATH': ROOT})

In [ ]:
subprocess.run([
    _sys.executable,
    os.path.join(ROOT, 'scripts', 'jobs', 'job3_scaling_sweep.py'),
    '--fineweb-records',  os.path.join(ROOT, 'outputs', 'fineweb_labeled.json'),
    '--marco-records',    os.path.join(ROOT, 'outputs', 'marco_labeled.json'),
    '--scifact-records',  os.path.join(ROOT, 'outputs', 'scifact_sweep_records.json'),
    '--fiqa-records',     os.path.join(ROOT, 'outputs', 'fiqa_sweep_records.json'),
    '--scifact-data-path', os.path.join(ROOT, 'outputs', 'beir_data', 'scifact'),
    '--fiqa-data-path',   os.path.join(ROOT, 'outputs', 'beir_data', 'fiqa'),
    '--output-dir',       os.path.join(ROOT, 'outputs', 'week9', 'scaling'),
], check=True, cwd=ROOT, env={**_os.environ, 'PYTHONPATH': ROOT})

## Fine-tuning Ablation (FineWeb → SciFact, k=10)

Trains on FineWeb, tests zero-shot on SciFact, then fine-tunes on 10 SciFact examples and measures recovery on the held-out queries.

In [ ]:
subprocess.run([
    _sys.executable,
    os.path.join(ROOT, 'scripts', 'jobs', 'job4_paper_bottleneck_ablation.py'),
    '--fineweb-queries-path', os.path.join(ROOT, 'outputs', 'fineweb_queries.jsonl'),
    '--scifact-data-dir',     os.path.join(ROOT, 'outputs', 'beir_data'),
    '--output-path',          os.path.join(ROOT, 'outputs', 'week9', 'paper_bottleneck_ablation_scifact.json'),
    '--k-shot', '10',
], check=True, cwd=ROOT, env={**_os.environ, 'PYTHONPATH': ROOT})

In [ ]:
import json

with open(os.path.join(ROOT, 'outputs', 'week9', 'paper_bottleneck_ablation_scifact.json')) as f:
    ab = json.load(f)

print("=== Fine-tuning Recovery Summary ===")
print(f"  Oracle nDCG@10 (all queries):      {ab['oracle_ndcg_at_10']:.3f}")
print(f"  Fixed-lambda nDCG@10 (all):        {ab['fixed_ndcg_at_10']:.3f}")
print(f"  MLP zero-shot nDCG@10 (all):       {ab['mlp_zero_shot_ndcg_at_10']:.3f}")
print()
print(f"  --- held-out queries (n={ab['held_out_eval_queries']}) ---")
print(f"  Oracle nDCG@10:                    {ab['oracle_held_out_ndcg_at_10']:.3f}")
print(f"  Zero-shot nDCG@10:                 {ab['mlp_zero_shot_held_out_ndcg_at_10']:.3f}")
print(f"  {ab['k_shot']}-shot fine-tuned nDCG@10:          {ab['mlp_k_shot_ft_ndcg_at_10']:.3f}")
print(f"  Recovery: {ab['recovery_pct']:.1f}% of oracle gap")